In [1]:
pip install contractions

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import re 
import string 
import contractions

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab', quiet=True)

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rajguru\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Rajguru\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rajguru\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Rajguru\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
df = pd.read_csv("data/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
stop_word = set(stopwords.words('english'))
lemmetizer = WordNetLemmatizer()

In [5]:
def preprocess_text(text):
    # Lowercase
    text = text.lower()

    # Expand contractions
    text = contractions.fix(text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove email IDs
    text = re.sub(r"\S+@\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove repeated characters
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_word]

    # Lemmatization
    tokens = [lemmetizer.lemmatize(word) for word in tokens]

    # Join tokens back into a sentence
    return " ".join(tokens)


In [6]:
df["clean_review"] = df["review"].apply(preprocess_text)

In [7]:
df[['review','clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


In [8]:
df.to_csv("./data/IMDB_Cleaned.csv", index=False)

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "I love this movie",
    "This movie is fantastic",
    "I hate this movie",
    "This film is boring",
    "I love this fantastic film"
]

count_vectorizer = CountVectorizer()
X_bow = count_vectorizer.fit_transform(documents)


In [10]:
X_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 18 stored elements and shape (5, 8)>

In [11]:
X_bow.toarray()

array([[0, 0, 0, 0, 0, 1, 1, 1],
       [0, 1, 0, 0, 1, 0, 1, 1],
       [0, 0, 0, 1, 0, 0, 1, 1],
       [1, 0, 1, 0, 1, 0, 0, 1],
       [0, 1, 1, 0, 0, 1, 0, 1]])

In [12]:
count_vectorizer.get_feature_names_out()

array(['boring', 'fantastic', 'film', 'hate', 'is', 'love', 'movie',
       'this'], dtype=object)

In [13]:
bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=count_vectorizer.get_feature_names_out()
)

bow_df.head()


,boring,fantastic,film,hate,is,love,movie,this
0,0,0,0,0,0,1,1,1
1,0,1,0,0,1,0,1,1
2,0,0,0,1,0,0,1,1
3,1,0,1,0,1,0,0,1
4,0,1,1,0,0,1,0,1


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [15]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(documents)


In [16]:
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 18 stored elements and shape (5, 8)>

In [17]:
X_tfidf.toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.70050458, 0.58148208, 0.41372929],
       [0.        , 0.57373967, 0.        , 0.        , 0.57373967,
        0.        , 0.47625576, 0.33885989],
       [0.        , 0.        , 0.        , 0.77253573, 0.        ,
        0.        , 0.51737618, 0.36811741],
       [0.62883263, 0.        , 0.50733821, 0.        , 0.50733821,
        0.        , 0.        , 0.29964212],
       [0.        , 0.54645401, 0.54645401, 0.        , 0.        ,
        0.54645401, 0.        , 0.32274454]])

In [18]:
tfidf_vectorizer.get_feature_names_out()

array(['boring', 'fantastic', 'film', 'hate', 'is', 'love', 'movie',
       'this'], dtype=object)

In [19]:
# Apply TF-IDF to the actual cleaned IMDB reviews
imdb_tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_imdb_tfidf = imdb_tfidf_vectorizer.fit_transform(df["clean_review"])

print("TF-IDF matrix shape:", X_imdb_tfidf.shape)
print("Sample features:", imdb_tfidf_vectorizer.get_feature_names_out()[:20])


TF-IDF matrix shape: (50000, 5000)
Sample features: ['aaron' 'abandoned' 'abc' 'ability' 'able' 'abrupt' 'absence' 'absent'
 'absolute' 'absolutely' 'absurd' 'absurdity' 'abuse' 'abused' 'abusive'
 'abysmal' 'academy' 'accent' 'accept' 'acceptable']
